In [1]:
!pip install h2o

In [2]:
import os

import h2o
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from h2o.automl import H2OAutoML
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler

In [3]:
USE_GOOGLE_COLAB = False

if USE_GOOGLE_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    CSV_PATH = os.path.join(os.getcwd(), 'drive', 'MyDrive', 'DSE_260A', 'data', 'cleaned_data')
    CHALLENGE_DATA_PATH = os.path.join(os.getcwd(), 'drive', 'MyDrive', 'DSE_260A', 'data', 'cleaned_data')
    SUBMISSION_PATH = os.getcwd()
else:
    CSV_PATH = '..'
    CHALLENGE_DATA_PATH = '../cleaned_data'
    SUBMISSION_PATH = 'submission'

#Final Data cleaning for Task 4.1

In [4]:
df_train = pd.read_csv(CSV_PATH + "/merged_data/combined.csv")
challenge_participants = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_participants_cleaned.csv')
challenge_hai = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_hai_cleaned.csv')
challenge_transcriptomics = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_transcriptomics_cleaned.csv')
df_challenge = challenge_hai.merge(challenge_participants, on='participant_id', how='inner')
df_challenge = df_challenge.merge(challenge_transcriptomics, on='participant_id', how='inner')

In [5]:
#drop any d28 HAI columns which are not the target column (HAI_H1N1 A/Victoria/4897/2022_d28)
df_train = df_train.drop(columns=[c for c in df_train.columns if c.endswith('_d28') and c.startswith(
    'HAI') and c != 'HAI_H1N1 A/Victoria/4897/2022_d28'])

#filter out participants who do not have a baseline measurement for H1N1 A/Victoria/4897/2022 at day 0
df_train = df_train[df_train['HAI_H1N1 A/Victoria/4897/2022_d28'].isna() == False]

#filter out columns which are all NaN
df_train = df_train.dropna(axis=1, how='all')

In [6]:
df_train['DELTA'] = df_train['HAI_H1N1 A/Victoria/4897/2022_d28'] - df_train['HAI_H1N1 A/Victoria/4897/2022_d0']
df_train.drop(columns=['HAI_H1N1 A/Victoria/4897/2022_d28'], inplace=True)

df_train['DELTA'].value_counts()

DELTA
 0.0    152
 1.0     20
 2.0     10
-1.0      6
 3.0      6
-3.0      4
 3.0      3
-2.0      3
 4.0      3
 7.0      2
 4.0      1
-4.0      1
 1.0      1
Name: count, dtype: int64

In [7]:
df_train.head()

,participant_id,PART_biological_sex,PART_arm_name,PART_age,HAI_H1N1 A/Brisbane/2/2018_d0,HAI_H1N1 A/California/7/2009_d0,HAI_H1N1 A/Guangdong-Maonan/SWL1536/2019_d0,HAI_H1N1 A/Victoria/2570/2019_d0,HAI_H1N1 A/Victoria/4897/2022_d0,HAI_H3N2 A/Darwin/9/2021_d0,HAI_H3N2 A/Hong Kong/2671/2019_d0,HAI_H3N2 A/Kansas/14/2017_d0,HAI_H3N2 A/Singapore/INFIMH-160019/2016_d0,HAI_H3N2 A/South Australia/34/2019_d0,HAI_H3N2 A/Tasmania/503/2020_d0,HAI_Vic B/Austria/1359417/2021_d0,HAI_Vic B/Colorado/6/2017_d0,HAI_Vic B/Washington/2/2019_d0,HAI_Yam B/Phuket/3073/2013_d0,DELTA
3466,2023_UGA.ID_002,male,Standard Fluzone,46.0,6.321928,6.321928,2.321928,6.321928,2.321928,2.321928,6.321928,5.321928,4.321928,5.321928,7.321928,6.321928,6.321928,7.321928,5.321928,0.0
3467,2023_UGA.ID_014,female,High Dose Fluzone,68.0,3.321928,3.321928,2.321928,2.321928,2.321928,4.321928,4.321928,3.321928,4.321928,6.321928,8.321928,3.321928,2.321928,2.321928,6.321928,0.0
3468,2023_UGA.ID_039,female,Standard Fluzone,28.0,8.321928,7.321928,6.321928,3.321928,2.321928,4.321928,8.321928,6.321928,8.321928,9.321928,9.321928,2.321928,2.321928,2.321928,8.321928,0.0
3469,2023_UGA.ID_058,female,Standard Fluzone,63.0,4.321928,4.321928,2.321928,2.321928,2.321928,6.321928,7.321928,5.321928,7.321928,7.321928,8.321928,5.321928,3.321928,2.321928,6.321928,0.0
3470,2023_UGA.ID_065,female,Standard Fluzone,30.0,7.321928,8.321928,4.321928,5.321928,2.321928,3.321928,9.321928,5.321928,8.321928,7.321928,8.321928,4.321928,7.321928,3.321928,9.321928,0.0


In [8]:
#figure out which columns are common to challenge and train set

challenge_cols = set(list(df_challenge.columns))
train_cols = set(list(df_train.columns))

common_cols = challenge_cols.intersection(train_cols)
print(common_cols)

missing_from_chal = train_cols - common_cols
print(missing_from_chal)

{'HAI_H1N1 A/Victoria/4897/2022_d0', 'PART_arm_name', 'HAI_H3N2 A/Kansas/14/2017_d0', 'HAI_H1N1 A/Brisbane/2/2018_d0', 'HAI_Vic B/Colorado/6/2017_d0', 'HAI_H1N1 A/California/7/2009_d0', 'HAI_H3N2 A/Hong Kong/2671/2019_d0', 'PART_age', 'participant_id', 'HAI_H1N1 A/Victoria/2570/2019_d0', 'HAI_Vic B/Washington/2/2019_d0', 'HAI_H3N2 A/Darwin/9/2021_d0', 'HAI_H3N2 A/Singapore/INFIMH-160019/2016_d0', 'HAI_Vic B/Austria/1359417/2021_d0', 'PART_biological_sex', 'HAI_H3N2 A/South Australia/34/2019_d0', 'HAI_H3N2 A/Tasmania/503/2020_d0', 'HAI_H1N1 A/Guangdong-Maonan/SWL1536/2019_d0', 'HAI_Yam B/Phuket/3073/2013_d0'}
{'DELTA'}


In [9]:
df_train['PART_arm_name'] = df_train['PART_arm_name'].apply(lambda x: 1.0 if x == 'High Dose Fluzone' else 0.0)
df_train['PART_biological_sex'] = df_train['PART_biological_sex'].apply(lambda x: 1.0 if x == 'Male' else -1.0)

scaler = StandardScaler()
df_train['PART_age'] = scaler.fit_transform(df_train[['PART_age']])
df_train

,participant_id,PART_biological_sex,PART_arm_name,PART_age,HAI_H1N1 A/Brisbane/2/2018_d0,HAI_H1N1 A/California/7/2009_d0,HAI_H1N1 A/Guangdong-Maonan/SWL1536/2019_d0,HAI_H1N1 A/Victoria/2570/2019_d0,HAI_H1N1 A/Victoria/4897/2022_d0,HAI_H3N2 A/Darwin/9/2021_d0,HAI_H3N2 A/Hong Kong/2671/2019_d0,HAI_H3N2 A/Kansas/14/2017_d0,HAI_H3N2 A/Singapore/INFIMH-160019/2016_d0,HAI_H3N2 A/South Australia/34/2019_d0,HAI_H3N2 A/Tasmania/503/2020_d0,HAI_Vic B/Austria/1359417/2021_d0,HAI_Vic B/Colorado/6/2017_d0,HAI_Vic B/Washington/2/2019_d0,HAI_Yam B/Phuket/3073/2013_d0,DELTA
3466,2023_UGA.ID_002,-1.0,0.0,-0.487221,6.321928,6.321928,2.321928,6.321928,2.321928,2.321928,6.321928,5.321928,4.321928,5.321928,7.321928,6.321928,6.321928,7.321928,5.321928,0.0
3467,2023_UGA.ID_014,-1.0,1.0,0.747778,3.321928,3.321928,2.321928,2.321928,2.321928,4.321928,4.321928,3.321928,4.321928,6.321928,8.321928,3.321928,2.321928,2.321928,6.321928,0.0
3468,2023_UGA.ID_039,-1.0,0.0,-1.497674,8.321928,7.321928,6.321928,3.321928,2.321928,4.321928,8.321928,6.321928,8.321928,9.321928,9.321928,2.321928,2.321928,2.321928,8.321928,0.0
3469,2023_UGA.ID_058,-1.0,0.0,0.467096,4.321928,4.321928,2.321928,2.321928,2.321928,6.321928,7.321928,5.321928,7.321928,7.321928,8.321928,5.321928,3.321928,2.321928,6.321928,0.0
3470,2023_UGA.ID_065,-1.0,0.0,-1.385401,7.321928,8.321928,4.321928,5.321928,2.321928,3.321928,9.321928,5.321928,8.321928,7.321928,8.321928,4.321928,7.321928,3.321928,9.321928,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3679,2023_UGA.ID_750,-1.0,0.0,-1.609946,4.321928,4.321928,2.321928,2.321928,2.321928,2.321928,3.321928,2.321928,3.321928,4.321928,7.321928,2.321928,2.321928,2.321928,6.321928,0.0
3680,2023_UGA.ID_751,-1.0,0.0,-1.441538,7.321928,7.321928,5.321928,6.321928,2.321928,2.321928,8.321928,3.321928,4.321928,7.321928,8.321928,2.321928,2.321928,2.321928,5.321928,0.0
3681,2023_UGA.ID_753,-1.0,0.0,0.523233,8.321928,8.321928,7.321928,6.321928,5.321928,3.321928,9.321928,6.321928,7.321928,8.321928,8.321928,4.321928,4.321928,4.321928,7.321928,1.0
3682,2023_UGA.ID_754,-1.0,0.0,-0.880175,6.321928,5.321928,4.321928,2.321928,2.321928,2.321928,3.321928,2.321928,3.321928,2.321928,5.321928,4.321928,2.321928,2.321928,5.321928,7.0


#Data preprocessing for the challenge set:

In [10]:
df_challenge = df_challenge[list(common_cols)]
df_challenge['PART_arm_name'] = df_challenge['PART_arm_name'].apply(lambda x: 1.0 if x == 'High Dose Fluzone' else 0.0)
df_challenge['PART_biological_sex'] = df_challenge['PART_biological_sex'].apply(lambda x: 1.0 if x == 'Male' else -1.0)
df_challenge['PART_age'] = scaler.transform(df_challenge[['PART_age']])
df_challenge.head()

,HAI_H1N1 A/Victoria/4897/2022_d0,PART_arm_name,HAI_H3N2 A/Kansas/14/2017_d0,HAI_H1N1 A/Brisbane/2/2018_d0,HAI_Vic B/Colorado/6/2017_d0,HAI_H1N1 A/California/7/2009_d0,HAI_H3N2 A/Hong Kong/2671/2019_d0,PART_age,participant_id,HAI_H1N1 A/Victoria/2570/2019_d0,HAI_Vic B/Washington/2/2019_d0,HAI_H3N2 A/Darwin/9/2021_d0,HAI_H3N2 A/Singapore/INFIMH-160019/2016_d0,HAI_Vic B/Austria/1359417/2021_d0,PART_biological_sex,HAI_H3N2 A/South Australia/34/2019_d0,HAI_H3N2 A/Tasmania/503/2020_d0,HAI_H1N1 A/Guangdong-Maonan/SWL1536/2019_d0,HAI_Yam B/Phuket/3073/2013_d0
0,2.321928,1.0,3.321928,2.321928,2.321928,4.321928,4.321928,0.972323,2024_UGA.ID_077,2.321928,2.321928,2.321928,3.321928,4.321928,-1.0,6.321928,3.321928,2.321928,4.321928
1,4.321928,1.0,2.321928,8.321928,5.321928,9.321928,2.321928,0.579369,2024_UGA.ID_086,6.321928,4.321928,2.321928,2.321928,5.321928,-1.0,4.321928,3.321928,9.321928,5.321928
2,5.321928,0.0,7.321928,6.321928,3.321928,7.321928,7.321928,-0.655629,2024_UGA.ID_128,6.321928,3.321928,6.321928,7.321928,3.321928,-1.0,8.321928,7.321928,6.321928,5.321928
3,2.321928,1.0,4.321928,3.321928,2.321928,3.321928,5.321928,1.084595,2024_UGA.ID_170,5.321928,2.321928,3.321928,4.321928,3.321928,-1.0,5.321928,6.321928,3.321928,3.321928
4,2.321928,1.0,6.321928,5.321928,4.321928,3.321928,3.321928,0.860050,2024_UGA.ID_179,6.321928,4.321928,4.321928,2.321928,5.321928,-1.0,4.321928,3.321928,2.321928,7.321928
5,2.321928,0.0,6.321928,2.321928,5.321928,3.321928,7.321928,-0.655629,2024_UGA.ID_215,2.321928,4.321928,5.321928,6.321928,4.321928,-1.0,8.321928,8.321928,2.321928,3.321928
6,2.321928,0.0,3.321928,6.321928,7.321928,6.321928,4.321928,-0.543357,2024_UGA.ID_219,5.321928,7.321928,3.321928,2.321928,5.321928,-1.0,4.321928,4.321928,6.321928,6.321928
7,3.321928,1.0,3.321928,3.321928,4.321928,2.321928,3.321928,1.421413,2024_UGA.ID_275,3.321928,4.321928,2.321928,2.321928,4.321928,-1.0,3.321928,4.321928,5.321928,7.321928
8,4.321928,1.0,4.321928,6.321928,5.321928,2.321928,2.321928,0.916187,2024_UGA.ID_295,6.321928,4.321928,2.321928,2.321928,9.321928,-1.0,4.321928,2.321928,5.321928,4.321928
9,2.321928,1.0,2.321928,5.321928,3.321928,4.321928,2.321928,0.972323,2024_UGA.ID_296,3.321928,4.321928,2.321928,2.321928,4.321928,-1.0,4.321928,2.321928,4.321928,3.321928


In [11]:
df_train.to_csv(os.path.join(os.getcwd(), '4.1_train_temp.csv'), index=False)

# Task 4.1

**4.1 predict magnitude of antibody response - H1N1 A/Victoria/4897/2022 (D28)**
* Training Data: Demographics + Day 0 + Day 7 innate
* Assay: HAI
* Measure: Single strain titer
* Metric: Spearman correlation
* Full description: HAI titer for H1N1 A/Victoria/4897/2022 at Day 28

In [12]:
TARGET_COL = 'DELTA'  # task 4.1

## Task 4.1: BASELINE LINNEAR REGRESSION MODEL
Magnitude refers to the strength of the antibody response against a single specific vaccine strain.
We are predicting the HAI titer for H1N1 A/Victoria/4897/2022 at Day 28 post-vaccination.

In [13]:
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "26.0.1" 2026-04-21; OpenJDK Runtime Environment Homebrew (build 26.0.1); OpenJDK 64-Bit Server VM Homebrew (build 26.0.1, mixed mode, sharing)
  Starting server from /Users/chayan/anaconda3/envs/cmi-flu-prediction-challenge-capstone/lib/python3.13/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /var/folders/kp/4mhl7j_j6354gqcs12nt1y680000gn/T/tmpjuqdv7b2
  JVM stdout: /var/folders/kp/4mhl7j_j6354gqcs12nt1y680000gn/T/tmpjuqdv7b2/h2o_chayan_started_from_python.out
  JVM stderr: /var/folders/kp/4mhl7j_j6354gqcs12nt1y680000gn/T/tmpjuqdv7b2/h2o_chayan_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,02 secs
H2O_cluster_timezone:,America/Los_Angeles
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,2 months and 19 days
H2O_cluster_name:,H2O_from_python_chayan_6d36en
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,15.98 Gb
H2O_cluster_total_cores:,12
H2O_cluster_allowed_cores:,12
H2O_cluster_status:,"locked, healthy"


In [14]:
data = h2o.import_file(os.path.join(os.getcwd(), '4.1_train_temp.csv'))

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [15]:
x = [c for c in data.columns if c != 'participant_id']
y = TARGET_COL
x = [c for c in x if c != y]

In [16]:
train, test = data.split_frame(ratios=[.8], seed=1234)

In [ ]:
aml = H2OAutoML(max_models=100, seed=1, nfolds=5,
                keep_cross_validation_predictions=True,
                max_runtime_secs=5400)

aml.train(x=x, y=y, training_frame=train)

AutoML progress: |
15:23:56.42: AutoML: XGBoost is not available; skipping it.
15:23:56.43: _train param, Dropping bad and constant columns: [PART_biological_sex]
15:23:56.83: _train param, Dropping bad and constant columns: [PART_biological_sex]
15:23:56.83: _min_rows param, The dataset size is too small to split for min_rows=100.0: must have at least 200.0 (weighted) rows, but have only 169.0.
15:23:56.83: _train param, Dropping bad and constant columns: [PART_biological_sex]


15:23:56.266: _train param, Dropping bad and constant columns: [PART_biological_sex]
15:23:56.371: _train param, Dropping bad and constant columns: [PART_biological_sex]


15:23:56.485: _train param, Dropping bad and constant columns: [PART_biological_sex]
15:23:56.603: _train param, Dropping bad and constant columns: [PART_biological_sex]

████
15:23:57.156: _train param, Dropping bad and constant columns: [PART_biological_sex]

██
15:23:57.450: _train param, Dropping bad and constant columns: [PART_biologica

In [ ]:
lb = aml.leaderboard
print(lb.head(rows=lb.nrows))

In [ ]:
print(aml.leader)

In [ ]:
predictions = aml.leader.predict(test)

results = test[y].cbind(predictions["predict"])
results.columns = ["actual", "predicted"]

# Filter out rows where actual is NaN
results_clean = results[results["actual"].isna() == 0]

print(results_clean.head(20))

In [ ]:
actual_values = results_clean['actual'].as_data_frame().to_numpy()
predicted_values = results_clean['predicted'].as_data_frame().to_numpy()

rho_results, pval_results = spearmanr(actual_values, predicted_values)
print(f'Spearman (results_clean): {rho_results:.3f}  (p-value: {pval_results:.4f})')

plt.figure(figsize=(6, 5))
plt.scatter(actual_values, predicted_values, alpha=0.5, color='darkorange')
plt.plot([actual_values.min(), actual_values.max()], [actual_values.min(), actual_values.max()], 'r--', label='Perfect')
plt.xlabel('True log(HAI) at D28 (Results Clean)')
plt.ylabel('Predicted (Results Clean)')
plt.title(f'Task 4.1 — Actual vs. Predicted (Results Clean)\nSpearman ρ = {rho_results:.3f}')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Flag to toggle between log scale (1) and original scale (0) results
convert_to_original_scale = 0  # Set to 0 for original scale, 1 for log2 scale

# Get the Day 0 HAI values for the test set corresponding to the cleaned results
test_d0_hai_column_name = TARGET_COL.replace('DELTA', 'HAI_H1N1 A/Victoria/4897/2022_d0')

# Filter the original 'test' H2OFrame using the same condition that created results_clean.
# results_clean was created from `results[results["actual"].isna() == 0]`,
# where `results["actual"]` corresponds to `test[y]` for the relevant rows.
filtered_test_for_d0_h2o = test[test[y].isna() == 0]

# Now, extract the d0 HAI column from this filtered H2OFrame
test_d0_hai_h2o_filtered = filtered_test_for_d0_h2o[test_d0_hai_column_name]

# Convert it to a numpy array. The 'actual_values' and 'predicted_values' are already numpy arrays.
test_d0_hai = test_d0_hai_h2o_filtered.as_data_frame().values.reshape(-1, 1)

# Calculate actual HAI d28 values (log2 scale)
actual_hai_d28_log2 = test_d0_hai + actual_values

# Calculate predicted HAI d28 values (log2 scale)
predicted_hai_d28_log2 = test_d0_hai + predicted_values

if convert_to_original_scale == 0:
    # Convert to original HAI titer scale (exp2)
    display_actual_values = np.exp2(actual_hai_d28_log2)
    display_predicted_values = np.exp2(predicted_hai_d28_log2)
    scale_label = 'Original Scale'
    x_label = 'True HAI at D28'
    y_label = 'Predicted HAI at D28'
else:
    display_actual_values = actual_hai_d28_log2
    display_predicted_values = predicted_hai_d28_log2
    scale_label = 'Log2 Scale'
    x_label = 'True log2(HAI) at D28'
    y_label = 'Predicted log2(HAI) at D28'

print(f"First 5 actual HAI D28 values ({scale_label}):\n", display_actual_values[:5])
print(f"\nFirst 5 predicted HAI D28 values ({scale_label}):\n", display_predicted_values[:5])

# Plotting the HAI values
plt.figure(figsize=(6, 5))
plt.scatter(display_actual_values, display_predicted_values, alpha=0.5, color='green')
plt.plot([display_actual_values.min(), display_actual_values.max()],
         [display_actual_values.min(), display_actual_values.max()], 'r--', label='Perfect')
plt.xlabel(x_label)
plt.ylabel(y_label)
plt.title(f'Task 4.1 — Actual vs. Predicted HAI at D28 ({scale_label})')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
#calculate the spearman
rho_results, pval_results = spearmanr(display_actual_values, display_predicted_values)

print(f'Spearman: {rho_results}')

#Use AutoML model to predict challenge set

In [ ]:
challenge_hf = h2o.H2OFrame(df_challenge)
y_pred_challenge = aml.leader.predict(challenge_hf).as_data_frame()['predict']

results = pd.DataFrame({
    'participant_id': df_challenge['participant_id'].values,
    'Task_4.1': np.exp2(y_pred_challenge),
})

results = results.merge(df_challenge[['participant_id', 'HAI_H1N1 A/Victoria/4897/2022_d0']], on='participant_id',
                        how='inner')
results['Task_4.1'] = np.exp2(results['Task_4.1'] + results['HAI_H1N1 A/Victoria/4897/2022_d0'])
d0 = results.copy(deep=True)
results.drop(columns=['HAI_H1N1 A/Victoria/4897/2022_d0'], inplace=True)
results

In [ ]:
d0

In [ ]:
plt.figure(figsize=(15, 7))

# Plot predicted Task_4.1 values
plt.scatter(d0['participant_id'], d0['Task_4.1'], color='red', label='Predicted Task_4.1', alpha=0.7)

# Plot Day 0 HAI values (d0 value)
plt.scatter(d0['participant_id'], d0['HAI_H1N1 A/Victoria/4897/2022_d0'], color='blue', label='Day 0 HAI', alpha=0.7)

plt.xlabel('Participant ID')
plt.ylabel('Value')
plt.title('Predicted Task_4.1 vs. Day 0 HAI for Challenge Participants')
plt.xticks(rotation=90, fontsize=8)  # Rotate x-axis labels for readability
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
#output to csv
results.to_csv(SUBMISSION_PATH + '/task_4_1.csv', index=False)